# 02 - Data Understanding and Initial Exploration

## Objective

The objective of this notebook is to understand the structure, quality, and business meaning of the dataset after removing exact duplicates.

This includes:

- Validating data structure and data types
- Understanding missing values
- Interpreting key business columns
- Exploring relationships between day type and attendance remarks

This notebook focuses on foundational understanding before moving into deeper pattern analysis.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

print("Libraries imported successfully")

project_root = Path.cwd().parent
processed_path = project_root / "data" / "processed"

no_duplicate_file_path = processed_path / "combined_no_exact_duplicates.csv"

print("Project root:", project_root)
print("Processed path:", processed_path)
print("No exact duplicate file exists:", no_duplicate_file_path.exists())

if no_duplicate_file_path.exists():
    print("File size MB:", no_duplicate_file_path.stat().st_size / (1024 * 1024))

In [ ]:
df = pd.read_csv(no_duplicate_file_path, low_memory=False)

print("Data loaded successfully")
print("Shape:", df.shape)

df.head()

In [ ]:
df = pd.read_csv(no_duplicate_file_path, low_memory=False)

print("Data loaded successfully")
print("Shape:", df.shape)

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.isna().sum()


Missing value analysis shows that core attendance and hour-related columns are complete. 
CAREER_BAND has very few missing values and can be treated as Unknown if needed. 
LEAVE and TEMPORARY_CARD have high missing values, likely because these fields are populated only when leave or temporary card usage exists. These missing values should not be treated as data errors without business confirmation.


In [ ]:
print("LEAVE value counts:")
print(df["LEAVE"].value_counts(dropna=False).head(20))

print("\nTEMPORARY_CARD value counts:")
print(df["TEMPORARY_CARD"].value_counts(dropna=False).head(20))

LEAVE and TEMPORARY_CARD appear to be exception-based fields.

LEAVE is populated only when leave is recorded.
TEMPORARY_CARD is populated only when a temporary card is used.
Missing values in these columns likely indicate that the condition is not applicable, rather than missing data.

In [ ]:
print("WEEKOFF_HOLIDAY value counts:")
print(df["WEEKOFF_HOLIDAY"].value_counts(dropna=False))

print("\nREMARKS value counts:")
print(df["REMARKS"].value_counts(dropna=False))

In [ ]:
pd.crosstab(
    df["WEEKOFF_HOLIDAY"],
    df["REMARKS"],
    dropna=False
)

### WEEKOFF_HOLIDAY and REMARKS Pattern

We performed a grouped analysis between WEEKOFF_HOLIDAY and REMARKS to understand how attendance context varies across different day types.

In [ ]:
summary_weekoff_remarks = (
    df.groupby(["WEEKOFF_HOLIDAY", "REMARKS"])
      .size()
      .reset_index(name="count")
      .sort_values(["WEEKOFF_HOLIDAY", "count"], ascending=[True, False])
)

summary_weekoff_remarks.head(30)

The combination of WEEKOFF_HOLIDAY and REMARKS shows that the dataset contains both day-type and work-context information.

Workdays include multiple valid contexts such as Present, Home, Client Office, Leave, Training, and Business Travel.

Week-off and holiday records mostly align with expected statuses, but some PRESENT records are also observed, which will be investigated further in the next stage.

## Conclusion

This notebook helped establish the basic structure and meaning of the dataset.

Key observations:

- The dataset contains 7,225,967 rows and 21 columns after exact duplicate removal.
- Core attendance and hour-related columns are mostly complete.
- LEAVE and TEMPORARY_CARD appear to be exception-based fields.
- WEEKOFF_HOLIDAY represents the type of day: WORKDAY, WEEK-OFF, or HOLIDAY.
- REMARKS represents the attendance/work context such as PRESENT, Home, Client Office, Full Day Leave, Wipro Office, Training, and Business Travel.
- The combination of WEEKOFF_HOLIDAY and REMARKS reveals that workdays contain multiple valid work contexts.
- Some WEEK-OFF and HOLIDAY records also have PRESENT remarks, which will be investigated further in the next notebook.